# TFM Deteccion - Experimento completo

Entrena las 4 arquitecturas sobre CNNDetection completo (progan_train, todas las categorias) y evalua en E1 (progan_test) + E1b (CNN_synth_testset, cross-architecture). E2 (GenImage) queda desactivado por defecto; ver seccion final para habilitarlo.

**Requisitos:**
- Runtime Colab con GPU (T4 o A100 recomendado; en free tier el disco son ~113 GB).
- En `MyDrive/cnndetection-datasets/`: `progan_train.zip`, `progan_val.zip`, `progan_testset.zip`, `CNN_synth_testset.zip`.
- Cuenta de wandb (opcional; si no, el notebook corre en modo offline).

**Tiempo estimado**: ~2-4 h por modelo en T4 -> ~8-14 h total.

## 1. Drive + repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

!rm -rf /content/tfm_deteccion
!git clone https://github.com/manuelpalasanchez/tfm_deteccion.git
%cd /content/tfm_deteccion
!git pull

In [ ]:
!pip install -q -r requirements.txt

## 2. Extraccion de datos

Extrae 14 de las 20 categorias de `progan_train` (~48-50 GB extraido), dejando margen holgado sobre los ~113 GB de disco de Colab. Criterio de seleccion:

- **Se conservan** las clases pesadas y ricas en texturas (person, car, cat, dog, train) + una seleccion variada (airplane, bicycle, bird, boat, bus, chair, horse, motorbike, tvmonitor).
- **Se descartan** 6 categorias redundantes o menos discriminativas: cow y sheep (animales rurales similares), diningtable y sofa (texturas de interior solapadas con chair), pottedplant y bottle (objetos pequenos con menos variacion).

Luego se extraen `progan_val`, `progan_test` y `CNN_synth_testset` (~10 GB adicionales).

In [ ]:
import subprocess, os

drive_zips = '/content/drive/MyDrive/cnndetection-datasets'
cnn_root   = '/content/cnndetection'
os.makedirs(cnn_root, exist_ok=True)

# 14 categorias: mantiene las pesadas (person, car, cat, dog, train) + variedad.
# Descartadas: cow, sheep, diningtable, sofa, pottedplant, bottle.
cats_train = [
    'airplane', 'bicycle', 'bird', 'boat', 'bus',
    'car', 'cat', 'chair', 'dog', 'horse',
    'motorbike', 'person', 'train', 'tvmonitor',
]

os.makedirs(f'{cnn_root}/progan_train', exist_ok=True)
for cat in cats_train:
    print(f'[train] extrayendo {cat}...')
    subprocess.run(['unzip', '-q', '-n', f'{drive_zips}/progan_train.zip',
                    f'{cat}/*', '-d', f'{cnn_root}/progan_train'], check=True)

print('\nEspacio tras progan_train:')
!df -h /content | grep -v Filesystem

for zip_name, dst in [
    ('progan_val.zip',      f'{cnn_root}/progan_val'),
    ('progan_testset.zip',  cnn_root),
    ('CNN_synth_testset.zip', cnn_root),
]:
    print(f'extrayendo {zip_name}...')
    os.makedirs(dst, exist_ok=True)
    subprocess.run(['unzip', '-q', '-n', f'{drive_zips}/{zip_name}', '-d', dst], check=True)

# Si progan_testset se descomprime como 'progan_testset' en vez de 'progan_test', renombrar
if os.path.isdir(f'{cnn_root}/progan_testset') and not os.path.isdir(f'{cnn_root}/progan_test'):
    os.rename(f'{cnn_root}/progan_testset', f'{cnn_root}/progan_test')
    print('renombrado progan_testset -> progan_test')

print('\nEstructura de /content/cnndetection:')
!ls {cnn_root}
!df -h /content | grep -v Filesystem

## 3. Patch de base.yaml

Ajusta roots y output.base_dir sin tocar los configs de modelo. E1b habilitado (cross-architecture). E2 y E3 desactivados (ver seccion final).

In [ ]:
import yaml, pathlib

cfg_path = pathlib.Path('configs/base.yaml')
cfg = yaml.safe_load(cfg_path.read_text())

cfg['data']['train']['root']      = '/content/cnndetection'
cfg['data']['val']['root']        = '/content/cnndetection'
cfg['data']['eval_e1']['root']    = '/content/cnndetection'
cfg['data']['eval_e1']['split']   = 'test'
cfg['data']['eval_e1b']['root']   = '/content/cnndetection'
cfg['data']['eval_e1b']['enabled'] = True
cfg['data']['eval_e2']['enabled']  = False
cfg['data']['eval_e3']['enabled']  = False
cfg['data']['num_workers']        = 2

cfg['training']['epochs']     = 8
cfg['training']['batch_size'] = 128
cfg['training']['scheduler']['T_max'] = 8

cfg['output']['base_dir'] = '/content/drive/MyDrive/tfm-checkpoints'
cfg['wandb']['enabled'] = True

cfg_path.write_text(yaml.dump(cfg))
print(yaml.dump(cfg))

## 4. wandb (opcional)

Si no tienes cuenta wandb o no quieres tracking, salta esta celda y modifica el patch anterior con `cfg['wandb']['enabled'] = False`.

In [ ]:
!wandb login

## 5. Sanity check del dataset

In [ ]:
import sys
sys.path.insert(0, '/content/tfm_deteccion')
from data.cnndetection_dataset import CNNDetectionDataset
from data.transforms import get_eval_transforms

for split in ('train', 'val', 'test'):
    try:
        ds = CNNDetectionDataset(root='/content/cnndetection', split=split, transform=get_eval_transforms())
        print(f'{split}: {len(ds)} muestras, generadores={ds.generators}')
    except Exception as e:
        print(f'{split}: ERROR - {e}')

## 6. Train + evaluate (4 modelos)

`scripts/run_all.py` entrena cada modelo y, al terminar, corre la evaluacion (E1 + E1b) usando el mejor checkpoint de cada run.

Si Colab se te cae a mitad, los checkpoints estan en Drive; puedes relanzar con `--eval-only` para retomar solo la evaluacion.

In [ ]:
!python scripts/run_all.py

### Alternativa: modelo a modelo

Util si quieres iterar sobre uno solo o repetir la evaluacion sin reentrenar.

In [ ]:
modelo = 'resnet50'   # resnet50 | freqnet | vit | universalfakedetect
!python scripts/train.py --config configs/{modelo}.yaml

In [ ]:
import glob

modelo = 'resnet50'
ckpts = sorted(glob.glob(f'/content/drive/MyDrive/tfm-checkpoints/{modelo}_*/checkpoint_best.pth'))
assert ckpts, f'Sin checkpoint para {modelo}'
ckpt = ckpts[-1]
print('checkpoint:', ckpt)
!python scripts/evaluate.py --config configs/{modelo}.yaml --checkpoint "{ckpt}"

## 7. Resumen de metricas

In [ ]:
import glob, json
from pathlib import Path

rows = []
for modelo in ['resnet50', 'freqnet', 'vit', 'universalfakedetect']:
    runs = sorted(glob.glob(f'/content/drive/MyDrive/tfm-checkpoints/{modelo}_*'))
    if not runs:
        print(f'{modelo}: sin runs')
        continue
    last = Path(runs[-1])
    mpath = last / 'metrics.json'
    if not mpath.exists():
        print(f'{modelo}: {last.name} (sin metrics.json)')
        continue
    m = json.loads(mpath.read_text())
    for ronda, vals in m.items():
        rows.append((modelo, ronda, vals['auc_roc'], vals['average_precision'], vals['accuracy']))
        print(f"{modelo:<22} {ronda:<8} AUC={vals['auc_roc']:.4f} AP={vals['average_precision']:.4f} Acc={vals['accuracy']:.4f}")

try:
    import pandas as pd
    df = pd.DataFrame(rows, columns=['modelo', 'ronda', 'AUC', 'AP', 'Acc'])
    df.to_csv('/content/drive/MyDrive/tfm-checkpoints/resumen_metricas.csv', index=False)
    print('\nCSV guardado en Drive: tfm-checkpoints/resumen_metricas.csv')
except Exception as e:
    print('No se pudo guardar CSV:', e)

## 8. (Opcional) Anadir E2 - GenImage cross-generator

GenImage no esta replicado automaticamente: hay que **subir tu copia a Drive**. Opciones reales:

- **Fuente oficial**: repo `GenImage-Dataset/GenImage` en GitHub. Los enlaces apuntan a Baidu Netdisk (descarga lenta fuera de China) y a un mirror en Huggingface Hub (`datasets/hemg/genimage`, parcial - no todos los generadores). Descarga la particion que te interese (p. ej. Midjourney o Stable Diffusion v1.4), subela zippeada a tu Drive en `MyDrive/genimage-datasets/` y extrae aqui con el mismo patron de la seccion 2.

- **Estructura esperada** por `data/genimage_dataset.py`:
  ```
  /content/genimage/<generador>/val/{nature,ai}/*.png
  /content/genimage/<generador>/val/{0_real,1_fake}/*.png  # tambien soportado
  ```
  donde `<generador>` es uno de: midjourney, vqdm, wukong, stable_diffusion_v_1_4, stable_diffusion_v_1_5, glide, biggan, adm.

- **Activacion**: tras extraer, re-correr el patch de `base.yaml` con `cfg['data']['eval_e2']['enabled']=True` y `cfg['data']['eval_e2']['root']='/content/genimage'`, y relanzar solo la evaluacion con `python scripts/run_all.py --eval-only` (no hace falta reentrenar).

In [ ]:
# Plantilla para cuando tengas GenImage en Drive
# import subprocess, os, yaml, pathlib
# os.makedirs('/content/genimage', exist_ok=True)
# subprocess.run(['unzip', '-q', '-n', '/content/drive/MyDrive/genimage-datasets/midjourney.zip',
#                 '-d', '/content/genimage'], check=True)
# cfg_path = pathlib.Path('configs/base.yaml')
# cfg = yaml.safe_load(cfg_path.read_text())
# cfg['data']['eval_e2']['root'] = '/content/genimage'
# cfg['data']['eval_e2']['enabled'] = True
# cfg_path.write_text(yaml.dump(cfg))
# !python scripts/run_all.py --eval-only